# Model Experiments: CONTENT_ID 기반 여행 계획 생성 실험

Issue #7 v1 target: train a GRU recommender that predicts the next TourAPI `CONTENT_ID` from user/trip features and the route prefix.

This notebook covers data loading, coverage checks, preprocessing, baseline evaluation, GRU experiments, final metric comparison, and artifact saving.
* colab 커널 연동 후 학습시킴

In [1]:
!nvidia-smi

Sat Sep  5 11:06:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Import & Settings

In [2]:
from __future__ import annotations

import json
import math
import pickle
import random
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset


ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "processed"
ARTIFACT_DIR = ROOT / "artifacts" / "model_experiments"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = DATA_DIR / "total_input.csv"
SEQ_PATH = DATA_DIR / "total_travel_seq_with_contentid.csv"

SEED = 42
MIN_SEQUENCE_LEN = 2
MAX_SEQUENCE_LEN = 20
TOP_KS = (5, 10)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)
DEVICE

device(type='cuda')

## 2. Load Data & Coverage

In [5]:
input_df = pd.read_csv(INPUT_PATH, encoding="utf-8-sig")
seq_df = pd.read_csv(SEQ_PATH, encoding="utf-8-sig")

coverage = {
    "input_rows": len(input_df),
    "input_trips": input_df["trip_id"].nunique(),
    "sequence_rows": len(seq_df),
    "sequence_trips": seq_df["travel_id"].nunique(),
    "content_rows": int(seq_df["CONTENT_ID"].notna().sum()),
    "content_trips": int(seq_df.loc[seq_df["CONTENT_ID"].notna(), "travel_id"].nunique()),
}
coverage

{'input_rows': 7730,
 'input_trips': 7730,
 'sequence_rows': 71649,
 'sequence_trips': 7730,
 'content_rows': 4922,
 'content_trips': 2000}

In [6]:
seq_df["TOURAPI_MATCH_STATUS"].value_counts(dropna=False).head(20)

,count
TOURAPI_MATCH_STATUS,
out_of_scope_seoul,67232
matched,1253
excluded_endpoint,950
location_low_score,715
matched_location,654
location_api_error,600
missing_coord,174
low_score,46
location_no_result,18


## 3. Preprocess Sequence Samples

In [7]:
def normalize_content_id(series: pd.Series) -> pd.Series:
    return series.astype("string").str.replace(r"\.0$", "", regex=True)


matched_seq = seq_df.loc[seq_df["CONTENT_ID"].notna()].copy()
matched_seq["CONTENT_ID"] = normalize_content_id(matched_seq["CONTENT_ID"])
matched_seq = matched_seq.sort_values(
    ["travel_id", "day_index", "visit_order"], kind="mergesort"
)

trip_sequences = (
    matched_seq.groupby("travel_id")["CONTENT_ID"]
    .apply(lambda values: [str(value) for value in values if pd.notna(value)])
    .reset_index(name="content_sequence")
)
trip_sequences["sequence_len"] = trip_sequences["content_sequence"].map(len)
trip_sequences = trip_sequences.loc[trip_sequences["sequence_len"] >= MIN_SEQUENCE_LEN].copy()

model_input = input_df.merge(
    trip_sequences,
    left_on="trip_id",
    right_on="travel_id",
    how="inner",
    validate="one_to_one",
)

print(f"usable trips: {len(model_input):,}")
print(model_input["sequence_len"].describe())
model_input.head(3)

usable trips: 1,337
count    1337.000000
mean        3.185490
std         1.471282
min         2.000000
25%         2.000000
50%         3.000000
75%         4.000000
max        15.000000
Name: sequence_len, dtype: float64


,trip_id,area_code,trip_days,theme,has_child,has_elderly,has_disabled,companion_count,p0_age,p0_gender,...,p0_home,p0_preferred,p1_age,p1_gender,p1_style,p1_home,p1_preferred,travel_id,content_sequence,sequence_len
0,e_e000004,central,2,3,0,0,0,2,40,남,...,41,50110;42210;42170,NaN,NaN,NaN,NaN,NaN,e_e000004,"[264392, 264394]",2
1,e_e000023,central,2,4,0,0,0,2,60,남,...,28,46710;26350;50130,NaN,NaN,NaN,NaN,NaN,e_e000023,"[2381373, 775394, 126508]",3
2,e_e000040,central,3,3,0,0,0,1,30,남,...,41,50110;50130;11440,NaN,NaN,NaN,NaN,NaN,e_e000040,"[2495561, 128162]",2


## 4. Split, Vocabulary & Feature Encoder

In [8]:
train_trips, temp_trips = train_test_split(
    model_input["trip_id"], test_size=0.30, random_state=SEED, shuffle=True
)
valid_trips, test_trips = train_test_split(
    temp_trips, test_size=0.50, random_state=SEED, shuffle=True
)

split_map = {
    "train": set(train_trips),
    "valid": set(valid_trips),
    "test": set(test_trips),
}

assert split_map["train"].isdisjoint(split_map["valid"])
assert split_map["train"].isdisjoint(split_map["test"])
assert split_map["valid"].isdisjoint(split_map["test"])

train_df = model_input[model_input["trip_id"].isin(split_map["train"])].reset_index(drop=True)
valid_df = model_input[model_input["trip_id"].isin(split_map["valid"])].reset_index(drop=True)
test_df = model_input[model_input["trip_id"].isin(split_map["test"])].reset_index(drop=True)

split_sizes = {"train": len(train_df), "valid": len(valid_df), "test": len(test_df)}
split_sizes

{'train': 935, 'valid': 201, 'test': 201}

In [9]:
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

train_content_ids = sorted({cid for seq in train_df["content_sequence"] for cid in seq})
content_id_to_token = {PAD_TOKEN: 0, UNK_TOKEN: 1}
content_id_to_token.update({content_id: idx + 2 for idx, content_id in enumerate(train_content_ids)})
token_to_content_id = {idx: content_id for content_id, idx in content_id_to_token.items()}

numeric_features = [
    "trip_days",
    "has_child",
    "has_elderly",
    "has_disabled",
    "companion_count",
    "p0_age",
    "p1_age",
]
feature_columns = [column for column in input_df.columns if column != "trip_id"]
categorical_features = [column for column in feature_columns if column not in numeric_features]

feature_encoder = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="constant", fill_value="missing", keep_empty_features=True)),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]
            ),
            categorical_features,
        ),
    ],
    remainder="drop",
)

feature_encoder.fit(train_df[feature_columns])
user_feature_dim = int(feature_encoder.transform(train_df[feature_columns]).shape[1])

print(f"vocab size: {len(content_id_to_token):,}")
print(f"user feature dim: {user_feature_dim:,}")

vocab size: 762
user feature dim: 1,844


## 5. Dataset & DataLoader

In [10]:
def encode_sequence(content_sequence: list[str]) -> list[int]:
    return [content_id_to_token.get(str(content_id), content_id_to_token[UNK_TOKEN]) for content_id in content_sequence]


def make_next_place_samples(df: pd.DataFrame) -> list[dict]:
    encoded_features = feature_encoder.transform(df[feature_columns]).astype(np.float32)
    samples: list[dict] = []
    for row_idx, row in df.reset_index(drop=True).iterrows():
        tokens = encode_sequence(row["content_sequence"])
        for target_pos in range(1, len(tokens)):
            label = tokens[target_pos]
            if label == content_id_to_token[UNK_TOKEN]:
                continue
            prefix = tokens[max(0, target_pos - MAX_SEQUENCE_LEN):target_pos]
            samples.append(
                {
                    "trip_id": row["trip_id"],
                    "user_features": encoded_features[row_idx],
                    "prefix": prefix,
                    "label": label,
                }
            )
    return samples


class TravelSequenceDataset(Dataset):
    def __init__(self, samples: list[dict]) -> None:
        self.samples = samples

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> dict:
        return self.samples[idx]


def collate_batch(batch: list[dict]) -> dict[str, torch.Tensor]:
    user_features = torch.tensor(np.stack([item["user_features"] for item in batch]), dtype=torch.float32)
    lengths = torch.tensor([len(item["prefix"]) for item in batch], dtype=torch.long)
    max_len = int(lengths.max().item())
    sequences = torch.zeros((len(batch), max_len), dtype=torch.long)
    for idx, item in enumerate(batch):
        sequences[idx, : len(item["prefix"])] = torch.tensor(item["prefix"], dtype=torch.long)
    labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)
    return {
        "user_features": user_features,
        "sequences": sequences,
        "lengths": lengths,
        "labels": labels,
    }


train_samples = make_next_place_samples(train_df)
valid_samples = make_next_place_samples(valid_df)
test_samples = make_next_place_samples(test_df)

{
    "train_samples": len(train_samples),
    "valid_samples": len(valid_samples),
    "test_samples": len(test_samples),
}

{'train_samples': 2093, 'valid_samples': 363, 'test_samples': 354}

## 6. Baseline Metrics

In [11]:
def metric_at_k(labels: np.ndarray, scores: np.ndarray, ks: tuple[int, ...] = TOP_KS) -> dict[str, float]:
    metrics: dict[str, float] = {}
    order = np.argsort(-scores, axis=1)
    for k in ks:
        topk = order[:, :k]
        hits = topk == labels[:, None]
        metrics[f"recall@{k}"] = float(hits.any(axis=1).mean())

        reciprocal_ranks = []
        ndcgs = []
        for row_hits in hits:
            hit_positions = np.flatnonzero(row_hits)
            reciprocal_ranks.append(0.0 if len(hit_positions) == 0 else 1.0 / float(hit_positions[0] + 1))
            ndcgs.append(0.0 if len(hit_positions) == 0 else 1.0 / math.log2(float(hit_positions[0] + 2)))
        metrics[f"mrr@{k}"] = float(np.mean(reciprocal_ranks))
        metrics[f"ndcg@{k}"] = float(np.mean(ndcgs))
    return metrics


def baseline_scores(train_samples: list[dict], eval_samples: list[dict], vocab_size: int) -> np.ndarray:
    counts = np.ones(vocab_size, dtype=np.float32) * 1e-6
    counts[0] = -np.inf
    counts[1] = -np.inf
    for sample in train_samples:
        counts[sample["label"]] += 1.0
    return np.repeat(counts[None, :], repeats=len(eval_samples), axis=0)


valid_labels = np.array([sample["label"] for sample in valid_samples])
baseline_valid_scores = baseline_scores(train_samples, valid_samples, len(content_id_to_token))
baseline_valid_metrics = metric_at_k(valid_labels, baseline_valid_scores)
baseline_valid_metrics

{'recall@5': 0.15977961432506887,
 'mrr@5': 0.0881542699724518,
 'ndcg@5': 0.10601176222640478,
 'recall@10': 0.23415977961432508,
 'mrr@10': 0.0985242030696576,
 'ndcg@10': 0.1305071128170573}

## 7. GRU Model

In [12]:
class GRUContentRecommender(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        user_feature_dim: int,
        embedding_dim: int,
        hidden_dim: int,
        dropout: float,
    ) -> None:
        super().__init__()
        self.place_embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.user_projection = nn.Sequential(
            nn.Linear(user_feature_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
        )
        self.output = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, vocab_size),
        )

    def forward(self, user_features: torch.Tensor, sequences: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        embedded = self.place_embedding(sequences)
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        _, hidden = self.gru(packed)
        sequence_context = hidden[-1]
        user_context = self.user_projection(user_features)
        logits = self.output(torch.cat([sequence_context, user_context], dim=1))
        logits[:, 0] = -1e9
        logits[:, 1] = -1e9
        return logits

## 8. Train & Evaluate

In [13]:
@dataclass(frozen=True)
class ExperimentConfig:
    embedding_dim: int = 64
    hidden_dim: int = 96
    dropout: float = 0.2
    learning_rate: float = 1e-3
    batch_size: int = 64
    epochs: int = 8
    weight_decay: float = 1e-5


EXPERIMENT_GRID = [
    ExperimentConfig(embedding_dim=48, hidden_dim=64, dropout=0.2, learning_rate=1e-3, batch_size=64, epochs=6),
    ExperimentConfig(embedding_dim=64, hidden_dim=96, dropout=0.2, learning_rate=1e-3, batch_size=64, epochs=8),
    ExperimentConfig(embedding_dim=64, hidden_dim=128, dropout=0.3, learning_rate=7e-4, batch_size=64, epochs=8),
]

SMOKE_TEST = False
if SMOKE_TEST:
    EXPERIMENT_GRID = [ExperimentConfig(embedding_dim=16, hidden_dim=24, dropout=0.1, learning_rate=1e-3, batch_size=32, epochs=1)]
    train_samples = train_samples[:512]
    valid_samples = valid_samples[:256]
    test_samples = test_samples[:256]

In [14]:
def make_loader(samples: list[dict], batch_size: int, shuffle: bool) -> DataLoader:
    return DataLoader(
        TravelSequenceDataset(samples),
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=collate_batch,
    )


def run_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer | None = None) -> float:
    is_train = optimizer is not None
    model.train(is_train)
    loss_fn = nn.CrossEntropyLoss()
    total_loss = 0.0
    total_rows = 0
    for batch in loader:
        batch = {key: value.to(DEVICE) for key, value in batch.items()}
        with torch.set_grad_enabled(is_train):
            logits = model(batch["user_features"], batch["sequences"], batch["lengths"])
            loss = loss_fn(logits, batch["labels"])
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()
        rows = batch["labels"].size(0)
        total_loss += float(loss.item()) * rows
        total_rows += rows
    return total_loss / max(total_rows, 1)


@torch.no_grad()
def predict_scores(model: nn.Module, loader: DataLoader) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    scores = []
    labels = []
    for batch in loader:
        batch = {key: value.to(DEVICE) for key, value in batch.items()}
        logits = model(batch["user_features"], batch["sequences"], batch["lengths"])
        scores.append(logits.detach().cpu().numpy())
        labels.append(batch["labels"].detach().cpu().numpy())
    return np.concatenate(labels), np.concatenate(scores)


def train_one_config(config: ExperimentConfig) -> tuple[nn.Module, dict]:
    seed_everything(SEED)
    model = GRUContentRecommender(
        vocab_size=len(content_id_to_token),
        user_feature_dim=user_feature_dim,
        embedding_dim=config.embedding_dim,
        hidden_dim=config.hidden_dim,
        dropout=config.dropout,
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay,
    )
    train_loader = make_loader(train_samples, config.batch_size, shuffle=True)
    valid_loader = make_loader(valid_samples, config.batch_size, shuffle=False)
    history = []
    best_state = None
    best_recall = -1.0
    for epoch in range(1, config.epochs + 1):
        train_loss = run_epoch(model, train_loader, optimizer)
        valid_loss = run_epoch(model, valid_loader)
        labels, scores = predict_scores(model, valid_loader)
        valid_metrics = metric_at_k(labels, scores)
        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "valid_loss": valid_loss,
            **valid_metrics,
        }
        history.append(row)
        print({**asdict(config), **row})
        if valid_metrics["recall@10"] > best_recall:
            best_recall = valid_metrics["recall@10"]
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    summary = {"config": asdict(config), "history": history, "best_valid_recall@10": best_recall}
    return model, summary

In [15]:
experiment_results = []
best_model = None
best_result = None

for config in EXPERIMENT_GRID:
    model, result = train_one_config(config)
    experiment_results.append(result)
    if best_result is None or result["best_valid_recall@10"] > best_result["best_valid_recall@10"]:
        best_model = model
        best_result = result

pd.DataFrame(
    [
        {
            **result["config"],
            "best_valid_recall@10": result["best_valid_recall@10"],
            "last_valid_loss": result["history"][-1]["valid_loss"],
        }
        for result in experiment_results
    ]
).sort_values("best_valid_recall@10", ascending=False)

{'embedding_dim': 48, 'hidden_dim': 64, 'dropout': 0.2, 'learning_rate': 0.001, 'batch_size': 64, 'epochs': 6, 'weight_decay': 1e-05, 'epoch': 1, 'train_loss': 6.5680123241222255, 'valid_loss': 6.411321877776427, 'recall@5': 0.15151515151515152, 'mrr@5': 0.08930211202938475, 'ndcg@5': 0.10473872447808251, 'recall@10': 0.209366391184573, 'mrr@10': 0.09715881761336306, 'ndcg@10': 0.12358752790658913}
{'embedding_dim': 48, 'hidden_dim': 64, 'dropout': 0.2, 'learning_rate': 0.001, 'batch_size': 64, 'epochs': 6, 'weight_decay': 1e-05, 'epoch': 2, 'train_loss': 5.893089072532307, 'valid_loss': 5.700120129861122, 'recall@5': 0.18732782369146006, 'mrr@5': 0.1095959595959596, 'ndcg@5': 0.12896914858814584, 'recall@10': 0.23415977961432508, 'mrr@10': 0.11576041803314528, 'ndcg@10': 0.14403076310347776}
{'embedding_dim': 48, 'hidden_dim': 64, 'dropout': 0.2, 'learning_rate': 0.001, 'batch_size': 64, 'epochs': 6, 'weight_decay': 1e-05, 'epoch': 3, 'train_loss': 5.436871847580109, 'valid_loss': 5.5

,embedding_dim,hidden_dim,dropout,learning_rate,batch_size,epochs,weight_decay,best_valid_recall@10,last_valid_loss
1,64,96,0.2,0.0010,64,8,0.00001,0.388430,5.149632
2,64,128,0.3,0.0007,64,8,0.00001,0.366391,5.198930
0,48,64,0.2,0.0010,64,6,0.00001,0.319559,5.389662


## 9. Final Test Metrics

In [16]:
assert best_model is not None
assert best_result is not None

test_loader = make_loader(test_samples, best_result["config"]["batch_size"], shuffle=False)
test_labels, test_scores = predict_scores(best_model, test_loader)
gru_test_metrics = metric_at_k(test_labels, test_scores)

baseline_test_scores = baseline_scores(train_samples, test_samples, len(content_id_to_token))
baseline_test_metrics = metric_at_k(test_labels, baseline_test_scores)

metrics = {
    "coverage": coverage,
    "usable_trips": len(model_input),
    "split_sizes": split_sizes,
    "sample_sizes": {
        "train": len(train_samples),
        "valid": len(valid_samples),
        "test": len(test_samples),
    },
    "baseline_valid": baseline_valid_metrics,
    "baseline_test": baseline_test_metrics,
    "gru_test": gru_test_metrics,
    "experiments": experiment_results,
}

pd.DataFrame(
    [
        {"model": "popular_baseline", **baseline_test_metrics},
        {"model": "gru_contentid", **gru_test_metrics},
    ]
)

,model,recall@5,mrr@5,ndcg@5,recall@10,mrr@10,ndcg@10
0,popular_baseline,0.206215,0.137006,0.154142,0.268362,0.144787,0.173717
1,gru_contentid,0.338983,0.238136,0.263047,0.423729,0.249332,0.290334


## 10. Save Artifacts & Reload Check

In [17]:
best_config = best_result["config"]
checkpoint = {
    "model_state_dict": best_model.state_dict(),
    "model_class": "GRUContentRecommender",
    "vocab_size": len(content_id_to_token),
    "user_feature_dim": user_feature_dim,
    "config": best_config,
    "max_sequence_len": MAX_SEQUENCE_LEN,
    "pad_token_id": content_id_to_token[PAD_TOKEN],
    "unk_token_id": content_id_to_token[UNK_TOKEN],
}

torch.save(checkpoint, ARTIFACT_DIR / "best_gru_contentid.pt")

with open(ARTIFACT_DIR / "content_id_vocab.json", "w", encoding="utf-8") as fp:
    json.dump(
        {
            "content_id_to_token": content_id_to_token,
            "token_to_content_id": token_to_content_id,
            "pad_token": PAD_TOKEN,
            "unk_token": UNK_TOKEN,
        },
        fp,
        ensure_ascii=False,
        indent=2,
    )

with open(ARTIFACT_DIR / "feature_encoder.pkl", "wb") as fp:
    pickle.dump(feature_encoder, fp)

with open(ARTIFACT_DIR / "train_config.json", "w", encoding="utf-8") as fp:
    json.dump(
        {
            "seed": SEED,
            "min_sequence_len": MIN_SEQUENCE_LEN,
            "max_sequence_len": MAX_SEQUENCE_LEN,
            "top_ks": TOP_KS,
            "input_path": str(INPUT_PATH.relative_to(ROOT)),
            "sequence_path": str(SEQ_PATH.relative_to(ROOT)),
            "best_config": best_config,
        },
        fp,
        ensure_ascii=False,
        indent=2,
    )

with open(ARTIFACT_DIR / "metrics.json", "w", encoding="utf-8") as fp:
    json.dump(metrics, fp, ensure_ascii=False, indent=2)

sorted(path.name for path in ARTIFACT_DIR.iterdir())

['best_gru_contentid.pt',
 'content_id_vocab.json',
 'feature_encoder.pkl',
 'metrics.json',
 'train_config.json']

In [18]:
loaded_checkpoint = torch.load(ARTIFACT_DIR / "best_gru_contentid.pt", map_location=DEVICE)
loaded_model = GRUContentRecommender(
    vocab_size=loaded_checkpoint["vocab_size"],
    user_feature_dim=loaded_checkpoint["user_feature_dim"],
    embedding_dim=loaded_checkpoint["config"]["embedding_dim"],
    hidden_dim=loaded_checkpoint["config"]["hidden_dim"],
    dropout=loaded_checkpoint["config"]["dropout"],
).to(DEVICE)
loaded_model.load_state_dict(loaded_checkpoint["model_state_dict"])
loaded_model.eval()

sample_batch = collate_batch(test_samples[:4])
sample_batch = {key: value.to(DEVICE) for key, value in sample_batch.items()}
with torch.no_grad():
    sample_logits = loaded_model(
        sample_batch["user_features"],
        sample_batch["sequences"],
        sample_batch["lengths"],
    )
topk_token_ids = torch.topk(sample_logits, k=10, dim=1).indices.detach().cpu().numpy()
[[token_to_content_id[int(token_id)] for token_id in row] for row in topk_token_ids]

[['1796079',
  '2779504',
  '2003909',
  '2758150',
  '741658',
  '1434477',
  '129437',
  '130071',
  '125555',
  '130551'],
 ['1796079',
  '129437',
  '741658',
  '2613658',
  '2779504',
  '264390',
  '1434477',
  '130551',
  '2758150',
  '126508'],
 ['129437',
  '2613658',
  '1796079',
  '741658',
  '2480899',
  '264390',
  '2745995',
  '4054880',
  '2779504',
  '125555'],
 ['129437',
  '2613658',
  '2033483',
  '264394',
  '2745995',
  '3033047',
  '125555',
  '3076051',
  '4054880',
  '1921150']]